In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/kavach', exist_ok=True)
print("Drive ready. Put files in /content/drive/MyDrive/kavach/")

Mounted at /content/drive
Drive ready. Put files in /content/drive/MyDrive/kavach/


In [2]:
import os, glob

# 1. is Drive mounted?
print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))

# 2. what's in the top of your Drive?
print("\nTop of MyDrive:")
print(os.listdir("/content/drive/MyDrive")[:30])

# 3. find EVERY csv file anywhere in your Drive
print("\nAll CSV files in your Drive:")
for f in glob.glob("/content/drive/MyDrive/**/*.csv", recursive=True):
    print("  ", f)

Drive mounted: True

Top of MyDrive:
['Getting started.pdf', 'GDToT', 'Slayydrive', 'CXLT24AUG (Java Master Course | Crux August 2024)_1277804_1725556893275.pdf', 'IMG-20240905-WA0017.jpg', 'IMG-20240905-WA0018.jpg', 'IMG-20240905-WA0019.jpg', 'I was core member of my function management team.docx', 'Colab Notebooks', 'My_aadhar_.pdf.pdf', 'Iot-based-AI-System-for-Diabetic-Retinopathy-Screening-processed(lightpdf.com) final.pdf.pdf', 'final Developing-a-Portable-Retinal-Screening-Device planning-processed(lightpdf.com).pdf.pdf', 'undertaking .jpg', 'Google AI Studio', 'code in c++ (1).gdoc', 'code in c++.gdoc', 'Screenshot_20260201_180542_Blinkit.jpg', 'Diabetic  Retinopathy data', 'Screen Recording 2026-04-17 192707.mp4', 'Comprehensive Cloud Computing Systems.gslides', 'Priyanshu_Ranjan_ATS_Resume.pdf', 'PRIYANSHU RANJAN_Doc (8).pdf', 'final project.gsheet', 'GIT LINK.gsheet', 'Priyanshu_Ranjan_Kalvium_InstructionalDesign.docx', 'Priyanshu_Ranjan_Amazon_MLSS_SOP.pdf', 'Priyanshu_Ranj

In [3]:
# ================================================================
# Kavach - Multilingual Scam Detector using MuRIL
# MuRIL is a Google model that already knows 17 Indian languages
# ================================================================
!pip install -q transformers deep-translator

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from deep_translator import GoogleTranslator
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "google/muril-base-cased"
MAX_LEN = 128

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.4 MB/s eta 0:00:00


In [4]:
# ----------------------------------------------------------------
# STEP 1: translate our data into the other most-spoken languages
# ----------------------------------------------------------------
sms_base   = pd.read_csv("/content/drive/MyDrive/kavach/spamshield_clean (1).csv")
calls_base = pd.read_csv("/content/drive/MyDrive/kavach/digital_arrest_scams.csv")

# calls are very long, so cut them into small 60-word pieces
def cut_into_pieces(text, size=60):
    words = str(text).split()
    if len(words) <= size:
        return [text]
    return [" ".join(words[i:i+size]) for i in range(0, len(words), size)]

# every call piece is a scam
call_pieces = []
for _, row in calls_base.iterrows():
    for piece in cut_into_pieces(row["text"]):
        call_pieces.append({"text": piece, "label": "scam"})
call_df = pd.DataFrame(call_pieces)

# small balanced sample of english sms
sms_sample = pd.concat([
    sms_base[sms_base.label == "scam"].sample(200, random_state=1),
    sms_base[sms_base.label == "safe"].sample(200, random_state=1)
])[["text", "label"]]

# everything we will translate
to_translate = pd.concat([sms_sample, call_df], ignore_index=True)
print("Rows to translate per language:", len(to_translate))

# the 6 extra languages (english + hindi are already in our data)
translate_langs = ["bn", "te", "mr", "ta", "gu", "pa"]
#  bn=Bengali te=Telugu mr=Marathi ta=Tamil gu=Gujarati pa=Punjabi

all_rows = []
for lang in translate_langs:
    print("translating to:", lang)
    count = 0
    for _, row in to_translate.iterrows():
        try:
            new_text = GoogleTranslator(source="auto", target=lang).translate(str(row["text"])[:500])
            all_rows.append({"text": new_text, "label": row["label"], "type": "sms"})
        except:
            pass   # skip if a translation fails
        count += 1
        if count % 100 == 0:
            print("   ", lang, count, "/", len(to_translate))
    print("done:", lang, "| total rows so far:", len(all_rows))

pd.DataFrame(all_rows).to_csv("/content/drive/MyDrive/kavach/translated_scams.csv", index=False)
print("Saved translated_scams.csv with", len(all_rows), "rows")


Rows to translate per language: 1364
translating to: bn
    bn 100 / 1364
    bn 200 / 1364
    bn 300 / 1364
    bn 400 / 1364
    bn 500 / 1364
    bn 600 / 1364
    bn 700 / 1364
    bn 800 / 1364
    bn 900 / 1364
    bn 1000 / 1364
    bn 1100 / 1364
    bn 1200 / 1364
    bn 1300 / 1364
done: bn | total rows so far: 1364
translating to: te
    te 100 / 1364
    te 200 / 1364
    te 300 / 1364
    te 400 / 1364
    te 500 / 1364
    te 600 / 1364
    te 700 / 1364
    te 800 / 1364
    te 900 / 1364
    te 1000 / 1364
    te 1100 / 1364
    te 1200 / 1364
    te 1300 / 1364
done: te | total rows so far: 2728
translating to: mr
    mr 100 / 1364
    mr 200 / 1364
    mr 300 / 1364
    mr 400 / 1364
    mr 500 / 1364
    mr 600 / 1364
    mr 700 / 1364
    mr 800 / 1364
    mr 900 / 1364
    mr 1000 / 1364
    mr 1100 / 1364
    mr 1200 / 1364
    mr 1300 / 1364
done: mr | total rows so far: 4092
translating to: ta
    ta 100 / 1364
    ta 200 / 1364
    ta 300 / 1364
    ta 400 / 1

In [4]:
from deep_translator import GoogleTranslator
import pandas as pd

safe_seeds = [
    # family / home
    "Beta reached office safely, call me when free",
    "Dinner is ready, come home soon",
    "Please water the plants, I will be back by night",
    "Reached home safely, good night",
    "Can you buy milk and bread on your way home",
    "Mummy is calling you for lunch",
    "I will be late tonight, don't wait for dinner",
    "Papa reached the station, train is on time",
    "Pick up the kids from school at 3 pm",
    "Grandma is coming to visit this weekend",
    "Did you take your medicine today",
    "Let's go for a walk in the evening",
    "The house keys are under the mat",
    "Call me once you reach the airport",
    "Happy anniversary to both of you",
    # work / study
    "Meeting is at 4 pm tomorrow, please be on time",
    "Please send me the report by evening",
    "The presentation went really well today",
    "I will call you after the class gets over",
    "Don't forget to submit the assignment by Friday",
    "The office will be closed on Monday for the holiday",
    "Great job on the project everyone",
    "Can we reschedule the call to 5 pm",
    "Please review the document and share feedback",
    "The interview is scheduled for next Tuesday",
    "Team lunch tomorrow at 1 pm, please join",
    "I am working from home today",
    "The deadline has been extended to next week",
    "Thanks for covering my shift yesterday",
    "Let's catch up over coffee this week",
    # reminders / appointments
    "Doctor appointment is on Monday at 11 am",
    "Your dentist appointment is confirmed for 4 pm",
    "Reminder: pay the school fees before the 10th",
    "The car service is due next week",
    "Don't forget mom's birthday is on Sunday",
    "Yoga class starts at 6 am tomorrow",
    "Parents meeting at school on Saturday",
    "Remember to carry your umbrella, it might rain",
    "The electrician is coming at 11 am tomorrow",
    "Your library books are due next Monday",
    # legit transactions / services
    "Your Amazon order has shipped and arrives tomorrow",
    "Your food order is out for delivery",
    "Your cab is arriving in 5 minutes",
    "Thank you for shopping with us, visit again",
    "Your ticket has been booked successfully",
    "Your table reservation for 8 pm is confirmed",
    "Your parcel has been delivered, thank you",
    "Your subscription renews next month",
    "Your flight is on time, boarding at gate 12",
    "Your hotel booking is confirmed for two nights",
    # greetings / social
    "Good morning, have a great day ahead",
    "Happy birthday, have a wonderful day",
    "Congratulations on your new job",
    "Wishing you a happy Diwali to you and family",
    "Happy new year, may this year bring joy",
    "Get well soon, take care of yourself",
    "Congratulations on your wedding",
    "Have a safe journey, message when you reach",
    "So proud of you, well done",
    "Missing you, let's meet soon",
    "Thank you for the lovely gift",
    "Best of luck for your exam tomorrow",
    "Happy Holi, enjoy the colours",
    "Eid Mubarak to you and your family",
    "Merry Christmas and a happy new year",
    # casual chat
    "How was your day today",
    "Are we still on for lunch tomorrow",
    "Let's watch a movie this weekend",
    "The weather is really nice today",
    "Did you watch the match last night",
    "What are your plans for the weekend",
    "I am almost there, five more minutes",
    "Let's meet at the coffee shop at 5",
    "The food at that restaurant was amazing",
    "Can you share the photos from the trip",
    "See you at the gym in the morning",
    "Traffic is heavy, I might be a bit late",
    "Thanks for the ride today",
    "Let me know when you are free to talk",
    "Had a great time today, thank you",
    # everyday errands
    "Please pick up groceries on the way back",
    "The plumber fixed the tap this morning",
    "I paid the electricity bill online today",
    "The new sofa will be delivered on Sunday",
    "Let's plan the trip for next month",
    "The tailor said the dress will be ready Friday",
    "I dropped the car at the service center",
    "We are out of sugar, please get some",
    "The washing machine is working fine now",
    "Booked the tickets for the show tonight",
    # school / kids
    "The kids have a holiday tomorrow",
    "Please sign the school diary",
    "Sports day is next Wednesday",
    "The exam results will be out next week",
    "Homework is done, can I watch TV now",
    "Parent teacher meeting is at 10 am",
    "The school bus will come at 7:30",
    "Don't forget the water bottle and lunch box",
    "The annual function is on Friday evening",
    "He scored well in the maths test",
]

print("Total safe seeds:", len(safe_seeds))

langs = ["hi", "bn", "te", "mr", "ta", "gu", "pa"]
rows = [{"text": s, "label": "safe", "type": "sms"} for s in safe_seeds]   # keep English
for code in langs:
    print("->", code)
    for s in safe_seeds:
        try:
            rows.append({"text": GoogleTranslator(source="en", target=code).translate(s),
                         "label": "safe", "type": "sms"})
        except:
            pass

pd.DataFrame(rows).to_csv("/content/drive/MyDrive/kavach/safe_multilang.csv", index=False)
print("Saved", len(rows), "safe examples across 8 languages")

Total safe seeds: 100
-> hi
-> bn
-> te
-> mr
-> ta
-> gu
-> pa
Saved 800 safe examples across 8 languages


In [5]:
# ----------------------------------------------------------------
# STEP 2: load english sms + hindi calls + translated data
# ----------------------------------------------------------------
def load_data(file_name):
    data = pd.read_csv(file_name)

    text_col = "text" if "text" in data.columns else data.columns[0]

    label_col = None
    for c in ["label", "category", "class"]:
        if c in data.columns:
            label_col = c
            break

    new = pd.DataFrame()
    new["text"] = data[text_col].astype(str)

    # make every label just "scam" or "safe"
    if label_col is not None:
        clean = []
        for v in data[label_col].astype(str).str.lower():
            if v in ["safe", "ham", "normal", "0", "legit"]:
                clean.append("safe")
            else:
                clean.append("scam")
        new["label"] = clean
    else:
        new["label"] = "scam"

    new["type"] = data["type"] if "type" in data.columns else "sms"
    return new


sms   = load_data("/content/drive/MyDrive/kavach/spamshield_clean (1).csv")   # your actual path
calls = load_data("/content/drive/MyDrive/kavach/digital_arrest_scams.csv")
try:
    translated = load_data("/content/drive/MyDrive/kavach/translated_scams.csv")
except:
    translated = pd.DataFrame(columns=["text", "label", "type"])

# 👇 ADD THIS — the new safe multilingual data
safe_extra = load_data("/content/drive/MyDrive/kavach/safe_multilang.csv")

# 👇 include safe_extra in the combine
all_data = pd.concat([sms, calls, translated, safe_extra], ignore_index=True)
print("Total rows:", len(all_data))

Total rows: 136306


In [6]:
# ----------------------------------------------------------------
# STEP 3: cut long calls, remove junk, balance the data
# ----------------------------------------------------------------
def make_chunks(text, size=60):
    words = str(text).split()
    if len(words) <= size:
        return [text]
    return [" ".join(words[i:i+size]) for i in range(0, len(words), size)]

final = []
for i in range(len(all_data)):
    row = all_data.iloc[i]
    if row["type"] == "call":               # long calls -> pieces
        for piece in make_chunks(row["text"]):
            final.append([piece, row["label"]])
    else:                                    # sms stays as it is
        final.append([row["text"], row["label"]])

df = pd.DataFrame(final, columns=["text", "label"])
df = df[df["text"].str.len() > 8].drop_duplicates(subset=["text"])

# keep equal scam and safe so the model is not biased
scam = df[df["label"] == "scam"]
safe = df[df["label"] == "safe"]
smaller = min(len(scam), len(safe))

# IMPORTANT: cap the size so MuRIL trains in ~30 min, not hours
CAP = 5000
smaller = min(smaller, CAP)

df = pd.concat([scam.sample(smaller, random_state=42),
                safe.sample(smaller, random_state=42)]).sample(frac=1, random_state=42)

df["y"] = df["label"].apply(lambda x: 1 if x == "scam" else 0)
print(df["label"].value_counts())
print("Total training rows:", len(df))

label
safe    5000
scam    5000
Name: count, dtype: int64
Total training rows: 10000


In [7]:
# ----------------------------------------------------------------
# STEP 4: turn text into tokens MuRIL understands
# ----------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

X_train, X_test, y_train, y_test = train_test_split(
    df["text"].tolist(), df["y"].tolist(), test_size=0.2, random_state=42)

# gives the model one example at a time
class ScamDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        tokens = tokenizer(self.texts[i], truncation=True, padding="max_length",
                           max_length=MAX_LEN, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in tokens.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

train_data = ScamDataset(X_train, y_train)
test_data  = ScamDataset(X_test, y_test)

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_data, batch_size=32)

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

In [8]:
# ----------------------------------------------------------------
# STEP 5: train the model (uses GPU if available)
# ----------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()      # clear old gradients
        outputs = model(**batch)   # model predicts
        loss = outputs.loss        # how wrong it was
        loss.backward()            # learn from the mistake
        optimizer.step()           # update the model
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} - loss: {total_loss/len(train_loader):.4f}")

Using: cuda


pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

Epoch 1/3 - loss: 0.3774
Epoch 2/3 - loss: 0.1226
Epoch 3/3 - loss: 0.0698


In [9]:
# ----------------------------------------------------------------
# STEP 6: check how accurate the model is
# ----------------------------------------------------------------
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == batch["labels"]).sum().item()
        total += batch["labels"].size(0)
print("Test accuracy:", round(correct / total * 100, 2), "%")

Test accuracy: 94.5 %


In [10]:
# ----------------------------------------------------------------
# STEP 7: try it on real messages, then save the model
# ----------------------------------------------------------------
def check_message(msg):
    model.eval()
    tokens = tokenizer(msg, truncation=True, padding="max_length",
                       max_length=MAX_LEN, return_tensors="pt").to(device)
    with torch.no_grad():
        prob = torch.softmax(model(**tokens).logits, dim=1)[0]
    if prob[1] > prob[0]:
        return "SCAM (" + str(round(float(prob[1]) * 100, 1)) + "%)"
    else:
        return "SAFE (" + str(round(float(prob[0]) * 100, 1)) + "%)"

tests = [
    "This is CBI. You are under digital arrest. Transfer money now.",       # English
    "मैं दिल्ली सीबीआई से बात कर रहा हूं, आपके नाम पर एक पार्सल पकड़ा गया है",  # Hindi
    "আপনার নামে একটি পার্সেল ধরা পড়েছে, এখনই টাকা পাঠান",                  # Bengali
    "ఇది సిబిఐ. మీరు డిజిటల్ అరెస్ట్ లో ఉన్నారు. ఇప్పుడే డబ్బు పంపండి",         # Telugu
    "நண்பரே, நாளை மதியம் சாப்பாட்டுக்கு சந்திக்கலாமா?",                     # Tamil (safe)
]
for t in tests:
    print(check_message(t), "->", t[:45])

model.save_pretrained("/content/drive/MyDrive/kavach/kavach_muril")
tokenizer.save_pretrained("/content/drive/MyDrive/kavach/kavach_muril")
print("Model saved!")

SCAM (99.6%) -> This is CBI. You are under digital arrest. Tr
SCAM (92.2%) -> मैं दिल्ली सीबीआई से बात कर रहा हूं, आपके नाम
SCAM (99.5%) -> আপনার নামে একটি পার্সেল ধরা পড়েছে, এখনই টাকা
SCAM (99.1%) -> ఇది సిబిఐ. మీరు డిజిటల్ అరెస్ట్ లో ఉన్నారు. ఇ
SAFE (99.5%) -> நண்பரே, நாளை மதியம் சாப்பாட்டுக்கு சந்திக்கலா


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!


In [12]:
import requests, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# load your trained MuRIL model from Drive
MODEL_PATH = "/content/drive/MyDrive/kavach/kavach_muril"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device); model.eval()

TOKEN = "8742434256:AAHIot2bU7efSlyfeIAJYMtXAi6YU1NSowI"
API = f"https://api.telegram.org/bot{TOKEN}"

def analyze(msg):
    enc = tokenizer(msg, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        prob = torch.softmax(model(**enc).logits, -1)[0]
    if prob[1] > prob[0]:
        return (f"⚠️ SCAM ALERT — {float(prob[1])*100:.1f}% risk\n\n"
                "🚫 Do NOT pay money or share OTP/bank details.\n"
                "👮 Real police/CBI never arrest over a video call.\n"
                "🆘 Report: cybercrime.gov.in or call 1930")
    else:
        return (f"✅ Looks Safe — {float(prob[0])*100:.1f}% confidence\n\n"
                "Stay alert if it asks for money, OTP or personal info.")

def reply(chat_id, text):
    requests.post(f"{API}/sendMessage", json={"chat_id": chat_id, "text": text})

print("🛡️ Kavach bot (MuRIL — 8 languages) running... message it on Telegram!")
offset = None
while True:
    res = requests.get(f"{API}/getUpdates", params={"timeout":30, "offset":offset}).json()
    for u in res.get("result", []):
        offset = u["update_id"] + 1
        m = u.get("message", {})
        cid = m.get("chat", {}).get("id")
        txt = m.get("text", "")
        if not txt: continue
        if txt == "/start":
            reply(cid, "🛡️ Kavach — Fraud Shield (now in 8 Indian languages!)\n"
                       "Forward me any suspicious call text or message and I'll tell you if it's a scam.")
        else:
            reply(cid, analyze(txt))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

🛡️ Kavach bot (MuRIL — 8 languages) running... message it on Telegram!


KeyboardInterrupt: 